## Q1. Computer Screen Detection Using Hough Line Transformation

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT = Path('lab06_outputs')
OUT.mkdir(exist_ok=True)
rng = np.random.default_rng(6)

def show(image, title='', size=(12, 6)):
    plt.figure(figsize=size)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB) if image.ndim == 3 else image, cmap=None if image.ndim == 3 else 'gray')
    plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
LAB_IMAGE_PATH = None
EXPECTED_SLOTS = []

def create_lab_scene():
    image = np.full((570, 1040, 3), (34, 32, 30), np.uint8)
    cv2.rectangle(image, (0, 35), (1039, 540), (42, 41, 40), -1)
    expected = []
    for row, y in enumerate((95, 330)):
        for col, x in enumerate((50, 300, 550, 800)):
            box = (x, y, 190, 135)
            expected.append(box)
            cv2.rectangle(image, (x - 12, y + 150), (x + 205, y + 175), (70, 68, 65), -1)
            cv2.putText(image, f'PC-{row*4+col+1}', (x+50, y+174), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (205,205,205), 1)
            if (row, col) == (1, 1):
                continue
            status = 'OFF' if (row, col) in [(0, 2), (1, 3)] else 'ON'
            cv2.rectangle(image, (x, y), (x+190, y+135), (225, 226, 228), 4)
            cv2.rectangle(image, (x+7, y+7), (x+183, y+128), (6, 6, 6) if status == 'OFF' else (115, 75, 42), -1)
            if status == 'ON':
                cv2.rectangle(image, (x+13, y+15), (x+175, y+30), (230, 155, 105), -1)
                for i in range(6):
                    cv2.line(image, (x+22, y+44+i*12), (x+155-(i%3)*15, y+44+i*12), (125, 230, 235), 2)
    return image, expected

if LAB_IMAGE_PATH:
    lab_image = cv2.imread(str(LAB_IMAGE_PATH))
    if lab_image is None:
        raise FileNotFoundError(LAB_IMAGE_PATH)
else:
    lab_image, EXPECTED_SLOTS = create_lab_scene()
cv2.imwrite(str(OUT / 'lab_screens.png'), lab_image)
show(lab_image, 'Computer lab')

In [ ]:
screen_gray = cv2.cvtColor(lab_image, cv2.COLOR_BGR2GRAY)
screen_edges = cv2.Canny(screen_gray, 45, 130)
screen_lines = cv2.HoughLinesP(screen_edges, 1, np.pi/180, 55, minLineLength=65, maxLineGap=18)
line_mask = np.zeros_like(screen_gray)
if screen_lines is not None:
    for x1, y1, x2, y2 in screen_lines[:, 0]:
        if abs(x2-x1) < 6 or abs(y2-y1) < 6:
            cv2.line(line_mask, (x1,y1), (x2,y2), 255, 3)
line_mask = cv2.morphologyEx(line_mask, cv2.MORPH_CLOSE, np.ones((9,9), np.uint8))
print('Hough lines:', 0 if screen_lines is None else len(screen_lines))
show(line_mask, 'Hough lines', size=(11, 5))

In [ ]:
contours, _ = cv2.findContours(line_mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
candidates = []
for contour in contours:
    x, y, w, h = cv2.boundingRect(contour)
    if 140 <= w <= 240 and 95 <= h <= 175 and 1.2 <= w/max(h,1) <= 2.2:
        candidates.append((x,y,w,h))

def box_iou(a, b):
    ax,ay,aw,ah = a; bx,by,bw,bh = b
    overlap = max(0, min(ax+aw,bx+bw)-max(ax,bx)) * max(0,min(ay+ah,by+bh)-max(ay,by))
    return overlap / max(1,aw*ah+bw*bh-overlap)

def deduplicate(boxes):
    kept = []
    for box in sorted(boxes, key=lambda b:b[2]*b[3], reverse=True):
        if all(box_iou(box, old) < .40 for old in kept):
            kept.append(box)
    return sorted(kept, key=lambda b:(b[1], b[0]))

screens = deduplicate(candidates)
print('Detected screen boundaries:', len(screens))

In [ ]:
screen_result = lab_image.copy()
counts = {'ON': 0, 'OFF': 0, 'MISSING': 0}
for x,y,w,h in screens:
    roi = screen_gray[y+int(.2*h):y+int(.8*h), x+int(.2*w):x+int(.8*w)]
    status = 'ON' if roi.size and np.mean(roi) > 45 else 'OFF'
    counts[status] += 1
    color = (50,210,50) if status == 'ON' else (40,140,255)
    cv2.rectangle(screen_result, (x,y), (x+w,y+h), color, 3)
    cv2.putText(screen_result, status, (x+4,max(20,y-8)), cv2.FONT_HERSHEY_SIMPLEX, .65, color, 2)
for slot in EXPECTED_SLOTS:
    if not any(box_iou(slot, detected) > .35 for detected in screens):
        x,y,w,h = slot
        counts['MISSING'] += 1
        cv2.rectangle(screen_result, (x,y), (x+w,y+h), (0,0,255), 2)
        cv2.putText(screen_result, 'MISSING', (x+4,max(20,y-8)), cv2.FONT_HERSHEY_SIMPLEX,.6,(0,0,255),2)
cv2.imwrite(str(OUT / 'q1_screens_detected.png'), screen_result)
print(counts)
show(screen_result, 'Detected screens and status')

## Q2. Asset Tracking in a Computer Lab Using SIFT

In [ ]:
sift = cv2.SIFT_create(nfeatures=1800)
matcher = cv2.BFMatcher(cv2.NORM_L2)

def create_asset(name, seed):
    random = np.random.default_rng(seed)
    patch = np.full((170, 245, 3), (232,225,214), np.uint8)
    cv2.rectangle(patch, (5,5), (239,164), (45,45,45), 4)
    cv2.rectangle(patch, (15,15), (229,45), (80,100,170), -1)
    cv2.putText(patch, name, (24,37), cv2.FONT_HERSHEY_SIMPLEX,.72,(255,255,255),2)
    for i in range(105):
        x,y = random.integers(15,230),random.integers(52,158)
        color = tuple(int(v) for v in random.integers(25,210,3))
        cv2.circle(patch,(int(x),int(y)),int(random.integers(1,5)),color,-1)
    for i in range(9):
        x = 20+i*23
        cv2.line(patch,(x,65),(x+int(random.integers(-8,9)),147),(25,35,40),1)
    return patch

assets = {'MONITOR': create_asset('MONITOR', 11),
          'KEYBOARD': create_asset('KEYBOARD', 23),
          'CPU': create_asset('CPU', 37)}
fig, axes = plt.subplots(1,3,figsize=(12,3))
for ax,(name,photo) in zip(axes,assets.items()):
    ax.imshow(cv2.cvtColor(photo,cv2.COLOR_BGR2RGB)); ax.set_title(name); ax.axis('off')
plt.show()

In [ ]:
ASSET_SCENE_PATH = None
asset_scene = np.full((610, 1040, 3), (55,58,61), np.uint8)
asset_locations = [(50,80),(390,60),(680,270)]
for (name,patch),(x,y) in zip(assets.items(),asset_locations):
    h,w = patch.shape[:2]
    source = np.float32([[0,0],[w-1,0],[w-1,h-1],[0,h-1]])
    shift = {'MONITOR': [[0,6],[-7,0],[8,7],[0,0]],
             'KEYBOARD': [[3,0],[12,7],[0,3],[-4,10]],
             'CPU': [[0,4],[10,0],[5,8],[-7,5]]}[name]
    target = np.float32([[x+a+dx,y+b+dy] for (a,b),(dx,dy) in zip(source,shift)])
    transform = cv2.getPerspectiveTransform(source,target)
    warped = cv2.warpPerspective(patch,transform,(asset_scene.shape[1],asset_scene.shape[0]))
    mask = cv2.warpPerspective(np.full((h,w),255,np.uint8),transform,(asset_scene.shape[1],asset_scene.shape[0]))
    asset_scene[mask>0] = warped[mask>0]
if ASSET_SCENE_PATH:
    asset_scene = cv2.imread(str(ASSET_SCENE_PATH))
    if asset_scene is None:
        raise FileNotFoundError(ASSET_SCENE_PATH)
cv2.imwrite(str(OUT/'q2_asset_scene.png'),asset_scene)
show(asset_scene,'Computer assets')

In [ ]:
def sift_locate(reference, scene, min_matches=10, min_inliers=8):
    gray_ref = cv2.cvtColor(reference,cv2.COLOR_BGR2GRAY)
    gray_scene = cv2.cvtColor(scene,cv2.COLOR_BGR2GRAY)
    kp1,desc1 = sift.detectAndCompute(gray_ref,None)
    kp2,desc2 = sift.detectAndCompute(gray_scene,None)
    if desc1 is None or desc2 is None or len(desc2)<2:
        return None, 0, 0
    pairs = matcher.knnMatch(desc1,desc2,k=2)
    good = [first for first,second in pairs if first.distance < .73*second.distance]
    if len(good) < min_matches:
        return None,len(good),0
    src = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)
    H,inliers = cv2.findHomography(src,dst,cv2.RANSAC,4.5)
    num_inliers = 0 if inliers is None else int(inliers.sum())
    if H is None or num_inliers < min_inliers:
        return None,len(good),num_inliers
    h,w = reference.shape[:2]
    corners = cv2.perspectiveTransform(np.float32([[0,0],[w-1,0],[w-1,h-1],[0,h-1]]).reshape(-1,1,2),H)
    if not np.isfinite(corners).all():
        return None,len(good),num_inliers
    return np.int32(corners),len(good),num_inliers

In [ ]:
asset_result = asset_scene.copy()
inventory = {}
for name, reference in assets.items():
    corners, matches, inliers = sift_locate(reference,asset_scene)
    inventory[name] = {'detected': corners is not None, 'matches':matches, 'inliers':inliers}
    if corners is not None:
        cv2.polylines(asset_result,[corners],True,(50,230,60),3,cv2.LINE_AA)
        x,y = corners.reshape(-1,2).min(axis=0)
        cv2.putText(asset_result,name,(int(x),max(20,int(y)-10)),cv2.FONT_HERSHEY_SIMPLEX,.7,(50,230,60),2)
print(inventory)
cv2.imwrite(str(OUT/'q2_asset_tracking.png'),asset_result)
show(asset_result,'Asset inventory')

## Q3. Anomaly Detection in Sensor Data Using Wavelet Transformation

In [ ]:
samples = 1024
t = np.arange(samples)
sensor_rng = np.random.default_rng(42)
normal_signal = .18*np.sin(t/29) + .12*np.cos(t/60) + sensor_rng.normal(0,.21,samples)
sensor = normal_signal.copy()
true_anomalies = sensor_rng.choice(np.arange(30,samples-30),size=18,replace=False)
sensor[true_anomalies] += sensor_rng.choice([-1,1],size=18)*sensor_rng.uniform(1.9,2.6,size=18)
plt.figure(figsize=(13,3)); plt.plot(t,sensor,lw=.7); plt.title('Sensor data'); plt.show()

In [ ]:
def haar_denoise(values, levels=5):
    current = np.asarray(values,dtype=float).copy()
    details=[]
    for _ in range(levels):
        low = (current[::2]+current[1::2])/np.sqrt(2)
        high = (current[::2]-current[1::2])/np.sqrt(2)
        details.append(high)
        current = low
    sigma = np.median(np.abs(details[0]-np.median(details[0])))/.6745
    threshold = sigma*np.sqrt(2*np.log(len(values)))
    for high in reversed(details):
        high = np.sign(high)*np.maximum(np.abs(high)-threshold,0)
        reconstructed=np.empty(len(high)*2,dtype=float)
        reconstructed[::2]=(current+high)/np.sqrt(2)
        reconstructed[1::2]=(current-high)/np.sqrt(2)
        current=reconstructed
    return current

denoised = haar_denoise(sensor)
residual = sensor-denoised

In [ ]:
median = np.median(residual)
robust_sigma = 1.4826*np.median(np.abs(residual-median))
anomaly_threshold = 4.0*robust_sigma
predicted_anomalies = np.flatnonzero(np.abs(residual-median)>anomaly_threshold)
print('Detected anomalies:',len(predicted_anomalies))
print('Detected positions:',predicted_anomalies.tolist())
print('True injected anomalies:',len(true_anomalies))
print('Precision:',round(len(set(predicted_anomalies)&set(true_anomalies))/max(1,len(predicted_anomalies)),3))
print('Recall:',round(len(set(predicted_anomalies)&set(true_anomalies))/len(true_anomalies),3))

In [ ]:
fig,axes=plt.subplots(2,1,figsize=(13,7),sharex=True)
axes[0].plot(t,sensor,label='Sensor data',lw=.75)
axes[0].plot(t,denoised,label='Denoised signal',lw=1.5)
axes[0].set_title('Sensor data and denoised signal'); axes[0].legend()
axes[1].plot(t,residual,color='red',label='Residuals',lw=.7)
axes[1].scatter(predicted_anomalies,residual[predicted_anomalies],s=28,c='green',label='Anomalies',zorder=3)
axes[1].axhline(anomaly_threshold,ls='--',color='gray')
axes[1].axhline(-anomaly_threshold,ls='--',color='gray')
axes[1].set_title('Residuals and detected anomalies');axes[1].legend()
plt.tight_layout();plt.savefig(OUT/'q3_wavelet_anomalies.png',dpi=160);plt.show()

## Q4. Object Recognition Using SIFT (Video)

In [ ]:
VIDEO_SOURCE = None
reference_object = assets['MONITOR'].copy()
cv2.imwrite(str(OUT/'q4_reference_object.png'),reference_object)


def make_sift_video(path, frames=30):
    width,height=720,480
    writer=cv2.VideoWriter(str(path),cv2.VideoWriter_fourcc(*'MJPG'),10,(width,height))
    if not writer.isOpened():
        raise RuntimeError('Could not create the demo video')
    h,w=reference_object.shape[:2]
    source=np.float32([[0,0],[w-1,0],[w-1,h-1],[0,h-1]])
    for i in range(frames):
        canvas=np.full((height,width,3),(48,49,50),np.uint8)
        cv2.rectangle(canvas,(60,70),(650,420),(70,71,74),2)
        cx=220+i*8;cy=225+int(32*np.sin(i/6))
        angle=np.deg2rad(-15+30*i/(frames-1));scale=.75+.25*np.sin(i/7)**2
        offsets=np.float32([[-w/2,-h/2],[w/2,-h/2],[w/2,h/2],[-w/2,h/2]])*scale
        rotation=np.array([[np.cos(angle),-np.sin(angle)],[np.sin(angle),np.cos(angle)]],np.float32)
        destination=(offsets@rotation.T+np.float32([cx,cy])).astype(np.float32)
        M=cv2.getPerspectiveTransform(source,destination)
        warped=cv2.warpPerspective(reference_object,M,(width,height))
        mask=cv2.warpPerspective(np.full((h,w),255,np.uint8),M,(width,height))
        canvas[mask>0]=warped[mask>0]
        if 12<=i<=15:
            cv2.rectangle(canvas,(int(cx+12),int(cy-7)),(int(cx+72),int(cy+56)),(48,49,50),-1)
        writer.write(canvas)
    writer.release()

sample_video=OUT/'q4_sift_demo.avi'
if VIDEO_SOURCE is None:
    make_sift_video(sample_video)
    VIDEO_SOURCE=str(sample_video)
print('Input:',VIDEO_SOURCE)

In [ ]:
capture=cv2.VideoCapture(VIDEO_SOURCE)
if not capture.isOpened():
    raise RuntimeError(f'Cannot open video source: {VIDEO_SOURCE}')
video_fps=capture.get(cv2.CAP_PROP_FPS) or 10
video_w=int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
video_h=int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
output_video=OUT/'q4_sift_detected.avi'
writer=cv2.VideoWriter(str(output_video),cv2.VideoWriter_fourcc(*'MJPG'),video_fps,(video_w,video_h))
if not writer.isOpened():
    raise RuntimeError('Cannot create result video')
max_frames=120
processed=0; recognized=0; snapshots=[]

In [ ]:
while processed<max_frames:
    ok,frame=capture.read()
    if not ok:
        break
    polygon,good,inliers=sift_locate(reference_object,frame,min_matches=9,min_inliers=7)
    if polygon is not None:
        cv2.polylines(frame,[polygon],True,(0,255,0),3)
        cv2.putText(frame,f'Object detected ({inliers} inliers)',(15,32),cv2.FONT_HERSHEY_SIMPLEX,.75,(0,255,0),2)
        recognized+=1
    else:
        cv2.putText(frame,'No object detected',(15,32),cv2.FONT_HERSHEY_SIMPLEX,.75,(0,130,255),2)
    if processed%8==0:
        snapshots.append(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
    writer.write(frame)
    processed+=1
capture.release();writer.release()
print(f'Recognized in {recognized}/{processed} frames')
print('Saved:',output_video)

In [ ]:
fig,axes=plt.subplots(1,min(4,len(snapshots)),figsize=(15,4))
for ax,frame in zip(np.atleast_1d(axes),snapshots[:4]):
    ax.imshow(frame); ax.axis('off')
plt.show()

## Q5. Panoramic Image Stitching Using SIFT

In [ ]:
PANORAMA_IMAGE_PATHS = []

def create_panorama_inputs():
    h,w=460,1680
    pano=np.full((h,w,3),(197,211,221),np.uint8)
    rand=np.random.default_rng(91)
    for i in range(50):
        x=int(i*35)
        top=int(95+25*np.sin(i*.55))
        color=tuple(int(v) for v in rand.integers(45,185,size=3))
        cv2.rectangle(pano,(x,top),(x+25,h-15),color,-1)
        for j in range(4):
            cv2.rectangle(pano,(x+4,top+15+j*45),(x+12,top+26+j*45),(225,232,237),-1)
        cv2.putText(pano,str(i),(x+2,min(h-20,top+70)),cv2.FONT_HERSHEY_SIMPLEX,.35,(255,255,255),1)
    for i in range(270):
        x=int(rand.integers(0,w));y=int(rand.integers(0,h))
        color=tuple(int(v) for v in rand.integers(0,255,size=3))
        cv2.circle(pano,(x,y),int(rand.integers(2,9)),color,-1)
    cv2.line(pano,(0,440),(w,440),(8,9,13),6)
    return [pano[:,offset:offset+720].copy() for offset in (0,450,900)]

if PANORAMA_IMAGE_PATHS:
    pano_images=[cv2.imread(str(path)) for path in PANORAMA_IMAGE_PATHS]
    if any(image is None for image in pano_images):
        raise FileNotFoundError('One of PANORAMA_IMAGE_PATHS could not be read')
else:
    pano_images=create_panorama_inputs()
for i,photo in enumerate(pano_images):
    cv2.imwrite(str(OUT/f'q5_input_{i+1}.png'),photo)
fig,axes=plt.subplots(1,len(pano_images),figsize=(15,4))
for ax,photo in zip(np.atleast_1d(axes),pano_images):
    ax.imshow(cv2.cvtColor(photo,cv2.COLOR_BGR2RGB));ax.axis('off')
plt.show()

In [ ]:
def estimate_pair_homography(previous,current):
    kp1,d1=sift.detectAndCompute(cv2.cvtColor(previous,cv2.COLOR_BGR2GRAY),None)
    kp2,d2=sift.detectAndCompute(cv2.cvtColor(current,cv2.COLOR_BGR2GRAY),None)
    if d1 is None or d2 is None:
        raise ValueError('Not enough features for panorama stitching')
    matches=matcher.knnMatch(d2,d1,k=2)
    good=[a for a,b in matches if a.distance < .7*b.distance]
    if len(good)<10:
        raise ValueError(f'Insufficient feature matches: {len(good)}')
    src=np.float32([kp2[m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst=np.float32([kp1[m.trainIdx].pt for m in good]).reshape(-1,1,2)
    H,mask=cv2.findHomography(src,dst,cv2.RANSAC,4)
    if H is None or int(mask.sum())<8:
        raise ValueError('Could not find a reliable homography')
    return H,len(good),int(mask.sum())

In [ ]:
transforms=[np.eye(3)]
for i in range(1,len(pano_images)):
    homography,matches,inliers=estimate_pair_homography(pano_images[i-1],pano_images[i])
    transforms.append(transforms[-1]@homography)
    print(f'Pair {i} -> {i+1}: {matches} matches, {inliers} inliers')

In [ ]:
all_corners=[]
for image,T in zip(pano_images,transforms):
    h,w=image.shape[:2]
    corners=np.float32([[0,0],[w,0],[w,h],[0,h]]).reshape(-1,1,2)
    all_corners.append(cv2.perspectiveTransform(corners,T).reshape(-1,2))
points=np.vstack(all_corners)
x_min,y_min=np.floor(points.min(axis=0)).astype(int)
x_max,y_max=np.ceil(points.max(axis=0)).astype(int)
translate=np.array([[1,0,-x_min],[0,1,-y_min],[0,0,1]],dtype=float)
canvas_w,canvas_h=x_max-x_min,y_max-y_min
accumulator=np.zeros((canvas_h,canvas_w,3),np.float32)
weights=np.zeros((canvas_h,canvas_w),np.float32)
for image,T in zip(pano_images,transforms):
    warped=cv2.warpPerspective(image,translate@T,(canvas_w,canvas_h)).astype(np.float32)
    mask=cv2.warpPerspective(np.ones(image.shape[:2],np.uint8),translate@T,(canvas_w,canvas_h))>0
    accumulator[mask]+=warped[mask]
    weights[mask]+=1
panorama=np.uint8(np.clip(accumulator/np.maximum(weights[:,:,None],1),0,255))
cv2.imwrite(str(OUT/'q5_panorama.png'),panorama)
print('Panorama size:',panorama.shape[1],'x',panorama.shape[0])
show(panorama,'Stitched panorama',size=(16,5))

## Q6. Lane Detection Using Hough Line Transformation

In [ ]:
ROAD_IMAGE_PATH = None

def create_road_image():
    h,w=540,960
    road=np.full((h,w,3),(190,170,125),np.uint8)
    cv2.rectangle(road,(0,0),(w,245),(220,195,125),-1)
    cv2.fillPoly(road,[np.int32([(405,240),(555,240),(950,540),(10,540)])],(70,75,80))
    cv2.line(road,(425,285),(205,535),(248,249,250),11,cv2.LINE_AA)
    cv2.line(road,(535,285),(750,535),(248,249,250),11,cv2.LINE_AA)
    for y in (300,370,450):
        delta=(y-240)*.09
        cv2.line(road,(480-int(delta),y),(480+int(delta),min(535,y+45)),(230,222,210),5)
    return road

road_image=cv2.imread(str(ROAD_IMAGE_PATH)) if ROAD_IMAGE_PATH else create_road_image()
if road_image is None:
    raise FileNotFoundError(ROAD_IMAGE_PATH)
cv2.imwrite(str(OUT/'q6_road_input.png'),road_image)
show(road_image,'Road input')

In [ ]:
road_h,road_w=road_image.shape[:2]
gray=cv2.cvtColor(road_image,cv2.COLOR_BGR2GRAY)
blur=cv2.GaussianBlur(gray,(5,5),0)
lane_edges=cv2.Canny(blur,60,160)
mask=np.zeros_like(lane_edges)
region=np.int32([[(80,road_h),(410,int(.5*road_h)),(550,int(.5*road_h)),(road_w-80,road_h)]])
cv2.fillPoly(mask,region,255)
restricted=cv2.bitwise_and(lane_edges,mask)
lane_segments=cv2.HoughLinesP(restricted,1,np.pi/180,35,minLineLength=45,maxLineGap=50)
print('Hough line segments:',0 if lane_segments is None else len(lane_segments))
show(restricted,'Lane edges')

In [ ]:
left,right=[],[]
if lane_segments is not None:
    for x1,y1,x2,y2 in lane_segments[:,0]:
        if x1==x2: continue
        slope=(y2-y1)/(x2-x1)
        if not .5 < abs(slope) < 4.0: continue
        intercept=y1-slope*x1
        length=np.hypot(x2-x1,y2-y1)
        (left if slope<0 else right).append((slope,intercept,length))

def average_lane(lines,y_bottom,y_top):
    if not lines: return None
    values=np.array(lines,dtype=float)
    slope,intercept=np.average(values[:,:2],axis=0,weights=values[:,2])
    x_bottom=int((y_bottom-intercept)/slope)
    x_top=int((y_top-intercept)/slope)
    return x_bottom,y_bottom,x_top,y_top

lane_result=road_image.copy()
for line in (average_lane(left,road_h-1,int(.55*road_h)),average_lane(right,road_h-1,int(.55*road_h))):
    if line is not None:
        x1,y1,x2,y2=line
        cv2.line(lane_result,(x1,y1),(x2,y2),(0,255,0),10,cv2.LINE_AA)
cv2.imwrite(str(OUT/'q6_lane_detection.png'),lane_result)
show(lane_result,'Detected lane markings')

## Q7. Coins Detection and Counting Using Hough Circle Transformation

In [ ]:
COINS_IMAGE_PATH = None

def make_coins_image():
    image=np.full((540,900,3),(37,43,49),np.uint8)
    layout=[(110,110,48),(285,100,56),(475,100,40),(675,108,51),
            (170,300,55),(385,310,44),(585,300,54),(780,310,40),(450,460,44)]
    colors=[(135,182,206),(123,172,195),(80,153,189)]
    for i,(x,y,r) in enumerate(layout):
        cv2.circle(image,(x+3,y+5),r+3,(15,19,23),-1,cv2.LINE_AA)
        cv2.circle(image,(x,y),r,colors[i%3],-1,cv2.LINE_AA)
        cv2.circle(image,(x,y),r-4,(210,225,233),3,cv2.LINE_AA)
        cv2.circle(image,(x,y),int(.55*r),(95,120,160),2,cv2.LINE_AA)
        cv2.putText(image,str(i+1),(x-12,y+9),cv2.FONT_HERSHEY_SIMPLEX,.8,(40,60,90),2)
    return image

coins_image=cv2.imread(str(COINS_IMAGE_PATH)) if COINS_IMAGE_PATH else make_coins_image()
if coins_image is None:
    raise FileNotFoundError(COINS_IMAGE_PATH)
cv2.imwrite(str(OUT/'q7_coins_input.png'),coins_image)
show(coins_image,'Coins image')

In [ ]:
coins_gray=cv2.cvtColor(coins_image,cv2.COLOR_BGR2GRAY)
coins_blurred=cv2.medianBlur(coins_gray,5)
circles=cv2.HoughCircles(coins_blurred,cv2.HOUGH_GRADIENT,dp=1.2,minDist=85,
                         param1=115,param2=30,minRadius=25,maxRadius=65)
coins_result=coins_image.copy()
coin_count=0
if circles is not None:
    for idx,(cx,cy,r) in enumerate(np.round(circles[0]).astype(int),1):
        cv2.circle(coins_result,(cx,cy),r,(20,240,20),3)
        cv2.circle(coins_result,(cx,cy),3,(0,0,255),-1)
        cv2.putText(coins_result,str(idx),(cx-12,cy-12),cv2.FONT_HERSHEY_SIMPLEX,.8,(20,240,20),2)
        coin_count+=1
print('Coin count:',coin_count)
cv2.imwrite(str(OUT/'q7_coins_counted.png'),coins_result)
show(coins_result,'Detected coins')

## Q8. Smart Security System Using Boundary Detection

In [ ]:
SECURITY_VIDEO_SOURCE = None
SECURITY_ZONE = (260,100,545,350)

def make_security_video(path,n=55):
    size=(720,460)
    writer=cv2.VideoWriter(str(path),cv2.VideoWriter_fourcc(*'MJPG'),10,size)
    if not writer.isOpened():
        raise RuntimeError('Cannot create security input video')
    for i in range(n):
        frame=np.full((size[1],size[0],3),(70,76,80),np.uint8)
        cv2.rectangle(frame,(20,20),(700,440),(90,95,99),4)
        cv2.rectangle(frame,(150,70),(215,420),(40,45,53),-1)
        if i >= 10:
            x=25+(i-10)*12
            cv2.rectangle(frame,(x,220),(x+72,290),(20,105,230),-1)
            cv2.rectangle(frame,(x+9,229),(x+62,280),(40,160,250),2)
        writer.write(frame)
    writer.release()

security_demo=OUT/'q8_security_demo.avi'
if SECURITY_VIDEO_SOURCE is None:
    make_security_video(security_demo)
    SECURITY_VIDEO_SOURCE=str(security_demo)
print('Input:',SECURITY_VIDEO_SOURCE)

In [ ]:
security_capture=cv2.VideoCapture(SECURITY_VIDEO_SOURCE)
if not security_capture.isOpened():
    raise RuntimeError(f'Cannot open security source: {SECURITY_VIDEO_SOURCE}')
w=int(security_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
h=int(security_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps=security_capture.get(cv2.CAP_PROP_FPS) or 10
security_output=OUT/'q8_security_alarm.avi'
security_writer=cv2.VideoWriter(str(security_output),cv2.VideoWriter_fourcc(*'MJPG'),fps,(w,h))
if not security_writer.isOpened():
    raise RuntimeError('Cannot write security result video')
background_subtractor=cv2.createBackgroundSubtractorMOG2(history=100,varThreshold=36,detectShadows=False)
security_frames=[];alarm_frames=0;frame_number=0
x0,y0,x1,y1=SECURITY_ZONE

In [ ]:
while frame_number<200:
    ok,frame=security_capture.read()
    if not ok: break
    learning_rate=.3 if frame_number<10 else 0
    mask=background_subtractor.apply(frame,learningRate=learning_rate)
    mask=cv2.morphologyEx(mask,cv2.MORPH_OPEN,np.ones((4,4),np.uint8))
    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,np.ones((9,9),np.uint8))
    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    alarm=False
    for contour in contours:
        if cv2.contourArea(contour)<350: continue
        x,y,bw,bh=cv2.boundingRect(contour)
        overlap=max(0,min(x+bw,x1)-max(x,x0))*max(0,min(y+bh,y1)-max(y,y0))
        is_inside = overlap/max(1,bw*bh)>.15
        if is_inside and frame_number>=10: alarm=True
        cv2.rectangle(frame,(x,y),(x+bw,y+bh),(0,0,255) if is_inside else (0,230,240),2)
    cv2.rectangle(frame,(x0,y0),(x1,y1),(0,0,255) if alarm else (0,220,0),3)
    cv2.putText(frame,'ALARM: INTRUSION' if alarm else 'SECURE', (18,45),
                cv2.FONT_HERSHEY_SIMPLEX,.9,(0,0,255) if alarm else (0,230,0),3)
    alarm_frames+=int(alarm)
    if frame_number in (5,15,25,35,45):
        security_frames.append(frame.copy())
    security_writer.write(frame)
    frame_number+=1
security_capture.release();security_writer.release()
print(f'Alarm triggered in {alarm_frames} of {frame_number} frames')
print('Saved:',security_output)

In [ ]:
fig,axes=plt.subplots(1,len(security_frames),figsize=(16,4))
for ax,frame in zip(np.atleast_1d(axes),security_frames):
    ax.imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB));ax.axis('off')
plt.show()